In [1]:
!pip install pyspark

In [2]:
from google.colab import files

uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [3]:

# 1. Import libraries

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim

# 2. Create Spark Session

spark = SparkSession.builder \
    .appName("IMDbMovieReviews") \
    .getOrCreate()

# 3. Load the dataset

df = spark.read.csv(
    "IMDB Dataset.csv",
    header=True,
    inferSchema=True
)

# 4. Explore the dataset

print("Dataset Schema:")
df.printSchema()

print("First 5 Records:")
df.show(5, truncate=False)

# 5. Remove null and empty reviews

cleaned_df = df.filter(
    col("review").isNotNull() &
    (trim(col("review")) != "")
)

# 6. Count total reviews

total_reviews = cleaned_df.count()

print("Total Reviews:", total_reviews)

# 7. Count positive and negative reviews

cleaned_df.groupBy("sentiment").count().show()

# 8. Create review length column

transformed_df = cleaned_df.withColumn(
    "review_length",
    length(col("review"))
)

# 9. Display reviews having more than 500 characters

long_reviews = transformed_df.filter(
    col("review_length") > 500
)

print("Reviews greater than 500 characters:")

long_reviews.select(
    "review",
    "sentiment",
    "review_length"
).show(10, truncate=False)

# 10. Save cleaned and transformed dataset

transformed_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("cleaned_imdb_reviews")

print("Dataset saved successfully!")

# 11. Stop Spark

spark.stop()


Dataset Schema:
root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)

First 5 Records:
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------